# 02 · 샘플 기술/노하우 데이터 주입

**파이프라인 2/4** — 파이프라인을 테스트할 수 있도록 실제 Teams/Mail에 샘플 기술·노하우
메시지/메일을 **write 도구로 전송**합니다.

MCP 툴을 직접 고르는 대신 **에이전트(LLM 도구호출 루프)** 에게 자연어로 지시하면, 에이전트가
각 서버의 알맞은 write 툴(메일 전송 / Teams 게시)을 스스로 골라 실행합니다. 에이전트 로직은
아래 셋업 셀에 인라인되어 있습니다(외부 pipeline 패키지 없음).

> ⚠️ 실제 메일/메시지가 전송됩니다. 기본값은 **본인에게 보내기 / 본인과의 채팅**입니다.

## 셋업 (설정 + 인증 + LLM 에이전트)

In [ ]:
# ============================================================
# 셋업 — 이 셀 하나로 설정·인증 준비 (pipeline/*.py 없이 독립 실행)
# ============================================================
import os, json, asyncio
from pathlib import Path
from contextlib import AsyncExitStack

import msal
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# 1) 프로젝트 루트를 찾아 .env 로드 (.env.example/requirements.txt가 있는 폴더)
def find_project_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / '.env.example').is_file() or (base / 'requirements.txt').is_file():
            return base
    return Path.cwd()

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env')
TOKEN_CACHE = PROJECT_ROOT / '.token_cache.json'   # 로그인 1회 → 이후 노트북/웹앱이 재사용

# 2) 설정값 (.env에서 읽음)
TENANT_ID     = os.environ.get('TENANT_ID') or '00000000-0000-0000-0000-000000000000'
CLIENT_ID     = os.environ.get('CLIENT_ID', '')
CLIENT_SECRET = os.environ.get('CLIENT_SECRET', '')
AUTH_MODE     = os.environ.get('AUTH_MODE', 'device_code')   # device_code | client_credentials
AUTHORITY     = f'https://login.microsoftonline.com/{TENANT_ID}'

def _mcp_url(env_var, server_id):
    return (os.environ.get(env_var) or '').strip() or \
        f'https://agent365.svc.cloud.microsoft/agents/tenants/{TENANT_ID}/servers/{server_id}'

# 두 MCP 소스: Mail / Teams. 서버마다 delegated 스코프가 달라 토큰도 소스별로 1개씩.
SOURCES = {
    'mail':  {'label': 'Mail',  'url': _mcp_url('MAIL_MCP_SERVER_URL',  'mcp_MailTools')},
    'teams': {'label': 'Teams', 'url': _mcp_url('TEAMS_MCP_SERVER_URL', 'mcp_TeamsServer')},
}
for _s in SOURCES.values():
    _s['scopes'] = [f"{_s['url']}/.default"]

# 3) MSAL 인증 — 소스별 access token. device_code는 최초 1회만 로그인, 이후 캐시 재사용.
_cache = msal.SerializableTokenCache()
if TOKEN_CACHE.exists():
    _cache.deserialize(TOKEN_CACHE.read_text(encoding='utf-8'))

def _save_cache():
    if _cache.has_state_changed:
        TOKEN_CACHE.write_text(_cache.serialize(), encoding='utf-8')

def ensure_token(source_key: str) -> str:
    """소스 1개의 access token 확보 (없으면 device-code 로그인)."""
    assert CLIENT_ID, 'CLIENT_ID가 .env에 없습니다 (.env.example 참고).'
    scopes = SOURCES[source_key]['scopes']
    if AUTH_MODE == 'client_credentials':           # 앱 전용 토큰
        app = msal.ConfidentialClientApplication(
            CLIENT_ID, authority=AUTHORITY, client_credential=CLIENT_SECRET, token_cache=_cache)
        res = app.acquire_token_for_client(scopes=scopes)
    else:                                           # 위임(사용자) 로그인
        app = msal.PublicClientApplication(CLIENT_ID, authority=AUTHORITY, token_cache=_cache)
        accounts = app.get_accounts()
        res = app.acquire_token_silent(scopes, account=accounts[0]) if accounts else None
        if not (res and 'access_token' in res):     # 캐시에 없으면 device-code 로그인
            flow = app.initiate_device_flow(scopes=scopes)
            print(flow['message'])                   # https://microsoft.com/devicelogin + 코드 안내
            res = app.acquire_token_by_device_flow(flow)
    _save_cache()
    if not res or 'access_token' not in res:
        raise RuntimeError((res or {}).get('error_description', 'access token 획득 실패'))
    return res['access_token']

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TENANT_ID   :', TENANT_ID, '| CLIENT_ID set:', bool(CLIENT_ID), '| AUTH_MODE:', AUTH_MODE)
for _k, _s in SOURCES.items():
    print(f"  [{_k}] {_s['label']} -> {_s['url']}")

In [ ]:
# --- LLM + 에이전트: 여러 MCP 소스에 연결해 LLM이 도구를 호출하는 루프 ---
def tool_schema(tool):
    sc = getattr(tool, 'inputSchema', None) or {'type': 'object', 'properties': {}}
    return sc.model_dump() if hasattr(sc, 'model_dump') else sc

def content_to_text(result) -> str:
    parts = []
    for b in (getattr(result, 'content', None) or []):
        if getattr(b, 'type', None) == 'text':
            parts.append(getattr(b, 'text', ''))
        else:
            data = b.model_dump() if hasattr(b, 'model_dump') else b
            parts.append('```json\n' + json.dumps(data, ensure_ascii=False, indent=2, default=str) + '\n```')
    text = '\n'.join(parts)
    return f'ERROR: {text}' if getattr(result, 'isError', False) else text

def make_openai():
    """Azure OpenAI(키 또는 Entra 키리스) 또는 OpenAI 클라이언트를 만든다."""
    ep, dep = os.environ.get('AZURE_OPENAI_ENDPOINT', ''), os.environ.get('AZURE_OPENAI_DEPLOYMENT', '')
    if ep and dep:
        from openai import AzureOpenAI
        common = dict(azure_endpoint=ep, azure_deployment=dep,
                      api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2024-10-21'))
        key = os.environ.get('AZURE_OPENAI_API_KEY', '')
        if key:
            return {'client': AzureOpenAI(api_key=key, **common), 'model': dep}
        from azure.identity import DefaultAzureCredential, get_bearer_token_provider
        tp = get_bearer_token_provider(DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default')
        return {'client': AzureOpenAI(azure_ad_token_provider=tp, **common), 'model': dep}
    okey = os.environ.get('OPENAI_API_KEY', '')
    if okey:
        from openai import OpenAI
        return {'client': OpenAI(api_key=okey), 'model': os.environ.get('OPENAI_MODEL', 'gpt-4o-mini')}
    raise RuntimeError('LLM 미설정 — .env에 AZURE_OPENAI_* 또는 OPENAI_API_KEY를 넣으세요.')

async def run_agent(tokens_by_source: dict, user_message: str, max_iters: int = 8) -> dict:
    oa = make_openai()
    trace, source_errors = [], {}
    async with AsyncExitStack() as stack:
        sessions, tools, registry = {}, [], {}
        # 1) 각 소스에 연결 + 도구 목록을 OpenAI 함수 스펙으로 변환
        for key, token in tokens_by_source.items():
            try:
                r, w, _ = await stack.enter_async_context(
                    streamablehttp_client(SOURCES[key]['url'], headers={'Authorization': f'Bearer {token}'}))
                s = await stack.enter_async_context(ClientSession(r, w))
                await s.initialize()
                sessions[key] = s
                for t in (await s.list_tools()).tools or []:
                    fname = f'{key}__{t.name}'[:64]          # 소스 네임스페이스로 이름 고유화
                    registry[fname] = (key, t.name)
                    tools.append({'type': 'function', 'function': {
                        'name': fname,
                        'description': f"[{SOURCES[key]['label']}] {t.description or t.name}",
                        'parameters': tool_schema(t)}})
            except Exception as exc:
                source_errors[key] = str(exc)
        if not tools:
            return {'answer': '사용할 수 있는 도구가 없습니다. 연결 상태를 확인하세요.',
                    'trace': trace, 'source_errors': source_errors}
        # 2) LLM ↔ 도구 호출 루프
        messages = [
            {'role': 'system', 'content':
                'You are connected to Microsoft "Work IQ" MCP servers. Tool names are prefixed by '
                'source ("mail__"/"teams__"). Prefer calling a tool over guessing; resolve names to '
                'real IDs first and never invent IDs. Answer in the user\'s language, concisely.'},
            {'role': 'user', 'content': user_message}]
        for _ in range(max_iters):
            comp = await asyncio.to_thread(
                oa['client'].chat.completions.create,
                model=oa['model'], messages=messages, tools=tools, tool_choice='auto', temperature=0)
            msg = comp.choices[0].message
            m = {'role': 'assistant', 'content': msg.content or ''}
            if msg.tool_calls:
                m['tool_calls'] = [{'id': c.id, 'type': 'function', 'function': {
                    'name': c.function.name, 'arguments': c.function.arguments}} for c in msg.tool_calls]
            messages.append(m)
            if not msg.tool_calls:                      # 도구 호출이 없으면 최종 답변
                return {'answer': msg.content or '(no content)', 'trace': trace, 'source_errors': source_errors}
            for c in msg.tool_calls:                     # 모델이 고른 도구를 실제 세션으로 라우팅
                try:
                    args = json.loads(c.function.arguments or '{}')
                except Exception:
                    args = {}
                key, orig = registry.get(c.function.name, (None, None))
                if key is None:
                    out = f'ERROR: unknown tool {c.function.name}'
                else:
                    try:
                        out = content_to_text(await sessions[key].call_tool(orig, arguments=args))
                    except Exception as exc:
                        out = f'ERROR calling {orig}: {exc}'
                trace.append({'tool': orig, 'source': key, 'args': args, 'result': out})
                messages.append({'role': 'tool', 'tool_call_id': c.id, 'content': out[:8000]})
        return {'answer': '도구 호출 한도(max_iters) 초과.', 'trace': trace, 'source_errors': source_errors}

## 1. 토큰 확보 (01에서 캐시된 로그인 재사용)

In [ ]:
tokens = {}
for key in SOURCES:
    try:
        tokens[key] = ensure_token(key)
    except Exception as exc:
        print(f'{key} 토큰 실패: {exc}')
print('확보된 소스:', list(tokens.keys()))
make_openai()   # LLM 설정 확인 (미설정이면 여기서 에러)
print('LLM OK')

## 2. 보낼 샘플 콘텐츠 정의
실제 위키에 담길 법한 재사용 가능한 기술 지식/노하우 예시입니다. 자유롭게 편집하세요.

In [ ]:
MAIL_TARGET  = '나 자신(로그인한 사용자)의 메일함으로'      # 예: 'alice@contoso.com 에게'
TEAMS_TARGET = '나 자신과의 1:1 채팅(없으면 새로 생성)'        # 예: "'Platform' 팀의 'General' 채널"

SAMPLE_EMAILS = [
    {'subject': '[노하우] Postgres 커넥션 풀 튜닝 정리',
     'body': ('PgBouncer를 transaction 모드로 두고 app 쪽 pool_size는 CPU 코어수*2로 제한. '
              'idle_in_transaction_session_timeout=30s로 좀비 커넥션 방지. '
              'max_connections를 무작정 늘리기보다 PgBouncer로 멀티플렉싱하는 게 안정적이었음.')},
    {'subject': '[트러블슈팅] k8s 파드 CrashLoopBackOff 원인 추적법',
     'body': ('kubectl describe pod로 이벤트 확인 -> kubectl logs --previous로 직전 컨테이너 로그. '
              'OOMKilled면 requests/limits 재조정. readinessProbe 타임아웃이 잦으면 initialDelaySeconds 상향.')},
]

SAMPLE_TEAMS_MESSAGES = [
    ('[팁] GitHub Actions 캐시 히트율 올리기: actions/cache로 node_modules + ~/.npm 둘 다 '
     '키에 lockfile 해시를 포함시키면 히트율이 크게 올라감.'),
    ('[결정사항] 사내 로그 표준: 구조적 로깅(JSON) + trace_id 필수. 레벨은 ERROR/WARN/INFO만, '
     'DEBUG는 로컬 전용.'),
]
print('메일', len(SAMPLE_EMAILS), '건, Teams', len(SAMPLE_TEAMS_MESSAGES), '건 준비됨')

## 3. 메일 샘플 전송 (Mail MCP · 에이전트)

In [ ]:
if 'mail' in tokens:
    lines = [f"- 제목: {e['subject']}\n  본문: {e['body']}" for e in SAMPLE_EMAILS]
    instruction = (f'다음 이메일들을 각각 별도의 메일로 {MAIL_TARGET} 보내줘. '
                   '제목과 본문을 그대로 사용하고, 전송 후 각 메일의 결과(성공/ID)를 요약해줘.\n\n'
                   + '\n'.join(lines))
    result = await run_agent({'mail': tokens['mail']}, instruction)
    print(result['answer'])
    if result['source_errors']:
        print('오류:', result['source_errors'])
else:
    print('mail 토큰이 없어 건너뜀')

## 4. Teams 샘플 메시지 게시 (Teams MCP · 에이전트)

In [ ]:
if 'teams' in tokens:
    lines = [f'- {m}' for m in SAMPLE_TEAMS_MESSAGES]
    instruction = (f'다음 메시지들을 {TEAMS_TARGET} 각각 게시해줘. '
                   '대상을 먼저 조회해서 실제 ID로 게시하고, 게시 결과를 요약해줘.\n\n'
                   + '\n'.join(lines))
    result = await run_agent({'teams': tokens['teams']}, instruction)
    print(result['answer'])
    if result['source_errors']:
        print('오류:', result['source_errors'])
else:
    print('teams 토큰이 없어 건너뜀')

✅ 샘플이 전송되었습니다. 인덱싱 지연이 있을 수 있으니 잠시 후 **03_fetch_data**에서 조회하세요.